In [1]:
#mount

from google.colab import drive
drive.mount('/content/drive')

# Adjust these two paths to match where your zip lives in Drive
DRIVE_ZIP_PATH = "/content/drive/MyDrive/Thesis Documents/mendeley_raw_data.zip"
LOCAL_SOURCE_DIR = "/content/mendeley_raw_data/Raw Data"

!cp "{DRIVE_ZIP_PATH}" /content/
!unzip -q /content/mendeley_raw_data.zip -d /content/mendeley_raw_data

Mounted at /content/drive


In [2]:
#pipeline

import os
import cv2
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm

# ---- Config ----
TARGET_CLASSES = ["5. Motherboard", "1. Battery", "8. RAM", "11. WifiCard", "20. CPUFan"]
SOURCE_DIR = "/content/mendeley_raw_data/Laptop Components Image Dataset to Classify Different Components/Raw Data/Raw Data"

BASE_OUTPUT_DIR = "/content/structured_ewaste_pipeline"
SPLITS = ["train", "val", "test"]
BLUR_THRESHOLD = 0.0
MIN_CLASS_SIZE = 10
VAL_TEST_SIZE = 0.20
RANDOM_STATE = 42


def calculate_laplacian_variance(image_path):
    """Calculates the Laplacian variance to detect image blur.
    Returns (variance, read_ok) so blur and read-failure are distinguishable."""
    image = cv2.imread(image_path)
    if image is None:
        return 0.0, False
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var(), True


def build_directory_structure():
    """Creates the train, val, and test folders for the 5 target classes."""
    for split in SPLITS:
        for component in TARGET_CLASSES:
            os.makedirs(os.path.join(BASE_OUTPUT_DIR, split, component), exist_ok=True)


def process_and_split_dataset():
    """Executes the ETL pipeline: filters blur/unreadable, applies stratified splitting."""
    dataset_records = []
    rejected_blurry = 0
    rejected_unreadable = 0

    # 1. Extract and Transform (Filter Blur + unreadable files)
    for component in TARGET_CLASSES:
        component_dir = os.path.join(SOURCE_DIR, component)

        if not os.path.exists(component_dir):
            print(f"Directory not found for class: {component}")
            continue

        filenames = [f for f in os.listdir(component_dir)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        for filename in tqdm(filenames, desc=component):
            file_path = os.path.join(component_dir, filename)
            blur_variance, read_ok = calculate_laplacian_variance(file_path)

            if not read_ok:
                rejected_unreadable += 1
                continue

            if blur_variance >= BLUR_THRESHOLD:
                dataset_records.append({"file_path": file_path, "label": component})
            else:
                rejected_blurry += 1

    print(f"\nRejected (unreadable): {rejected_unreadable}")
    print(f"Rejected (blurry, var < {BLUR_THRESHOLD}): {rejected_blurry}")

    df = pd.DataFrame(dataset_records)

    if df.empty:
        print("Error: No images passed the cleaning phase.")
        return None

    # ---- Class-count guard before stratified split ----
    counts = df['label'].value_counts()
    print("\nClass counts after filtering:")
    print(counts)

    starved = counts[counts < MIN_CLASS_SIZE]
    if not starved.empty:
        print(f"\nWARNING: these classes have < {MIN_CLASS_SIZE} images after filtering "
              f"and may break stratified splitting:\n{starved}")
        return None

    # 2. Load (Stratified Splitting: 80% Train, 10% Val, 10% Test)
    train_df, temp_df = train_test_split(
        df, test_size=VAL_TEST_SIZE, stratify=df['label'], random_state=RANDOM_STATE
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, stratify=temp_df['label'], random_state=RANDOM_STATE
    )

    # 3. Route files to their respective directories
    partitions = [("train", train_df), ("val", val_df), ("test", test_df)]

    for split_name, partition_df in partitions:
        print(f"\nPopulating {split_name} partition ({len(partition_df)} images)...")
        for _, row in tqdm(partition_df.iterrows(), total=len(partition_df), desc=split_name):
            src = row['file_path']
            dest = os.path.join(BASE_OUTPUT_DIR, split_name, row['label'], os.path.basename(src))
            shutil.copy2(src, dest)

    print("\nETL Pipeline complete. Data is structured and ready for model ingestion.")
    return df


print("Initializing E-Waste Classification Pipeline...")
build_directory_structure()
df = process_and_split_dataset()

Initializing E-Waste Classification Pipeline...


5. Motherboard:   0%|          | 0/140 [00:00<?, ?it/s]

1. Battery:   0%|          | 0/140 [00:00<?, ?it/s]

8. RAM:   0%|          | 0/140 [00:00<?, ?it/s]

11. WifiCard:   0%|          | 0/140 [00:00<?, ?it/s]

...


Populating val partition (70 images)...


val:   0%|          | 0/70 [00:00<?, ?it/s]


Populating test partition (70 images)...


test:   0%|          | 0/70 [00:00<?, ?it/s]


ETL Pipeline complete. Data is structured and ready for model ingestion.


In [3]:
import tensorflow as tf
from tensorflow.keras import layers

# ---- Dataset Configurations ----
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 16

print("Loading datasets into TensorFlow...")

# 1. Load splits directly from the variables defined in Cell 2
train_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "train"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=True,
    seed=RANDOM_STATE
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "val"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    os.path.join(BASE_OUTPUT_DIR, "test"),
    labels="inferred",
    label_mode="categorical",
    class_names=TARGET_CLASSES,
    batch_size=BATCH_SIZE,
    image_size=IMAGE_SIZE,
    shuffle=False
)

# 2. Configure runtime memory caching and prefetching
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)

# 3. Dynamic Augmentation Layer (Applied only during training)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(factor=(-30.0 / 360.0, 30.0 / 360.0)),
    layers.RandomZoom(height_factor=(-0.15, 0.15)),
    layers.RandomTranslation(height_factor=(-0.10, 0.10),
                             width_factor=(-0.10, 0.10)),
    layers.RandomBrightness(factor=(-0.25, 0.25)),
    layers.RandomContrast(factor=0.20)
], name="e_waste_augmentation")

print("\nTensorFlow Data Pipeline and Augmentation Layer Ready.")

Loading datasets into TensorFlow...
Found 560 files belonging to 5 classes.
Found 70 files belonging to 5 classes.
Found 70 files belonging to 5 classes.

TensorFlow Data Pipeline and Augmentation Layer Ready.


In [4]:
import tensorflow.keras.backend as K
from tensorflow.keras.applications import DenseNet121, MobileNetV2
from tensorflow.keras.applications.densenet import preprocess_input as densenet_preprocess
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

# ---- 1. Custom Focal Loss Implementation ----
class CategoricalFocalLoss(tf.keras.losses.Loss):
    def __init__(self, gamma=2.0, alpha=0.25, name="categorical_focal_loss"):
        super(CategoricalFocalLoss, self).__init__(name=name)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        # Clip predictions to prevent log(0) numerical instability
        epsilon = K.epsilon()
        y_pred = tf.clip_by_value(y_pred, epsilon, 1.0 - epsilon)

        # Cross-entropy calculation
        cross_entropy = -y_true * tf.math.log(y_pred)

        # Focal Loss modulating factor: (1 - p_t)^gamma
        focal_weight = self.alpha * tf.math.pow(1.0 - y_pred, self.gamma)

        loss = focal_weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))

# ---- 2. Model Builder Functions ----
def build_densenet_model(name="DenseNet-121", apply_augmentation=True):
    inputs = layers.Input(shape=(224, 224, 3))

    # Augmentation & Domain Preprocessing
    x = data_augmentation(inputs) if apply_augmentation else inputs
    x = densenet_preprocess(x)

    # Frozen Base Feature Extractor
    base_model = DenseNet121(weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = False

    # Custom Classification Head
    x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.5, name="dropout_0.5")(x)
    outputs = layers.Dense(len(TARGET_CLASSES), activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)
    return model, base_model

def build_mobilenet_model(name="MobileNet-V2", apply_augmentation=True):
    inputs = layers.Input(shape=(224, 224, 3))

    x = data_augmentation(inputs) if apply_augmentation else inputs
    x = mobilenet_preprocess(x)

    base_model = MobileNetV2(weights="imagenet", include_top=False, input_tensor=x)
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D(name="gap")(base_model.output)
    x = layers.Dense(256, activation="relu", name="dense_256")(x)
    x = layers.Dropout(0.5, name="dropout_0.5")(x)
    outputs = layers.Dense(len(TARGET_CLASSES), activation="softmax", name="predictions")(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs, name=name)
    return model, base_model

print("Loss function and architectures ready for compilation.")

Loss function and architectures ready for compilation.


In [5]:
# ---- 1. Drive Checkpoint Configuration ----
CHECKPOINT_DIR = "/content/drive/MyDrive/Thesis Documents/Model_Checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

def get_training_callbacks(model_name):
    # Saves the best weights directly to your Google Drive
    checkpoint = tf.keras.callbacks.ModelCheckpoint(
        filepath=os.path.join(CHECKPOINT_DIR, f"{model_name}_best.keras"),
        monitor="val_loss",
        save_best_only=True,
        mode="min",
        verbose=1
    )
    # Halts training if the model stops improving for 5 epochs
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
    return [checkpoint, early_stopping]

# ---- 2. Compile DenseNet-121 ----
print("Instantiating DenseNet-121 for Phase 1 (Frozen Base)...")
densenet_model, densenet_base = build_densenet_model(name="DenseNet-121")

densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)

# ---- 3. Execute Phase 1 Training ----
INITIAL_EPOCHS = 15

print(f"\nStarting Phase 1 Training for {INITIAL_EPOCHS} epochs...")
densenet_history = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=INITIAL_EPOCHS,
    callbacks=get_training_callbacks("DenseNet-121")
)

Instantiating DenseNet-121 for Phase 1 (Frozen Base)...
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

Starting Phase 1 Training for 15 epochs...
Epoch 1/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.4437 - loss: 0.2718
Epoch 1: val_loss improved from None to 0.04433, saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_best.keras

Epoch 1: finished saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_best.keras
35/35 ━━━━━━━━━━━━━━━━━━━━ 140s 4s/step - accuracy: 0.6143 - loss: 0.1578 - val_accuracy: 0.8429 - val_loss: 0.0443
Epoch 2/15
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.8226 - loss: 0.0515
Epoch 2: val_loss improved from 0.04433 to 0.01566, saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_best.keras

Epoch 2: finished saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_best.keras
35/35 ━━━━━━━━━━━━━━━━━━━━ 113s 3s/s

In [6]:
# ---- 1. Unfreeze the Base Model ----
print("Unfreezing DenseNet-121 base layers for fine-tuning...")
densenet_base.trainable = True

# ---- 2. Recompile with a Micro-Learning Rate ----
# A much lower learning rate (1e-5 instead of 1e-3) is critical here
densenet_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss=CategoricalFocalLoss(gamma=2.0, alpha=0.25),
    metrics=["accuracy"]
)

# ---- 3. Execute Phase 2 Fine-Tuning ----
FINE_TUNE_EPOCHS = 10
TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

print(f"\nStarting Phase 2 Fine-Tuning for {FINE_TUNE_EPOCHS} additional epochs...")
history_fine = densenet_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=TOTAL_EPOCHS,
    initial_epoch=densenet_history.epoch[-1], # Resume exactly where Phase 1 left off
    callbacks=get_training_callbacks("DenseNet-121_FineTuned")
)

Unfreezing DenseNet-121 base layers for fine-tuning...

Starting Phase 2 Fine-Tuning for 10 additional epochs...
Epoch 15/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.6899 - loss: 0.1851 
Epoch 15: val_loss improved from None to 0.00428, saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_FineTuned_best.keras

Epoch 15: finished saving model to /content/drive/MyDrive/Thesis Documents/Model_Checkpoints/DenseNet-121_FineTuned_best.keras
35/35 ━━━━━━━━━━━━━━━━━━━━ 531s 12s/step - accuracy: 0.7089 - loss: 0.1521 - val_accuracy: 0.9714 - val_loss: 0.0043
Epoch 16/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.7891 - loss: 0.0767 
Epoch 16: val_loss did not improve from 0.00428
35/35 ━━━━━━━━━━━━━━━━━━━━ 430s 12s/step - accuracy: 0.8071 - loss: 0.0751 - val_accuracy: 0.9571 - val_loss: 0.0054
Epoch 17/25
35/35 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.8636 - loss: 0.0406 
Epoch 17: val_loss did not improve from 0.00428
35/35 ━━━━━━━

In [7]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

print("Evaluating DenseNet-121 on unseen Test Dataset...")
test_loss, test_accuracy = densenet_model.evaluate(test_ds, verbose=1)

print(f"\nFinal DenseNet-121 Test Accuracy: {test_accuracy * 100:.2f}%")
print(f"Final DenseNet-121 Test Loss: {test_loss:.4f}")

# ---- Generate Classification Report ----
print("\nGenerating Detailed Classification Report...")
test_labels = np.concatenate([y for x, y in test_ds], axis=0)
true_classes = np.argmax(test_labels, axis=1)

predictions = densenet_model.predict(test_ds, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)

report = classification_report(true_classes, predicted_classes, target_names=TARGET_CLASSES)
print("\n" + report)

Evaluating DenseNet-121 on unseen Test Dataset...
5/5 ━━━━━━━━━━━━━━━━━━━━ 14s 2s/step - accuracy: 0.9857 - loss: 0.0030

Final DenseNet-121 Test Accuracy: 98.57%
Final DenseNet-121 Test Loss: 0.0030

Generating Detailed Classification Report...

                precision    recall  f1-score   support

5. Motherboard       1.00      1.00      1.00        14
    1. Battery       1.00      0.93      0.96        14
        8. RAM       1.00      1.00      1.00        14
  11. WifiCard       1.00      1.00      1.00        14
    20. CPUFan       0.93      1.00      0.97        14

      accuracy                           0.99        70
     macro avg       0.99      0.99      0.99        70
  weighted avg       0.99      0.99      0.99        70

